# CAMUS Baseline — LITE U-Net (RL warm-start baseline)

> **LITE baseline.** This is a deliberately lightweight plain U-Net (`model: lite_unet`, ~0.5M params vs ~33M for the attention net). It is intentionally weaker so its errors are *systematic* — giving the DRL contour agents real headroom to refine. The attention Res-UNet (notebooks 03/04) is kept as the upper-bound competitor. All outputs are suffixed `_lite_unet` so they never overwrite the attention baseline's checkpoint/scores/masks.

Thin notebook: all logic lives in the `iteris/` package. This file just installs deps,
loads config, and calls high-level functions.

**Kaggle setup**
1. Add CAMUS dataset to inputs (path used below: `/kaggle/input/datasets/anfaalhossain/camus/CAMUS`)
2. Add the **iteris-package-2 (folder iteris-pkg)** dataset (`/kaggle/input/datasets/anfaalhossain/iteris-package-2/iteris-pkg`) — contains the `iteris/` package + `configs/` + `requirements.txt`
3. Run the first cell, **restart the kernel**, then **Run All** from cell 2 onward

**To switch datasets:** change the YAML path in Cell 2. Nothing else changes.

## 0 · Setup
Lightweight install — uses Kaggle's default numpy/torch/scipy versions (no conflicts).
Run this cell once, then **Run All** (no kernel restart required).

In [ ]:
import subprocess
from pathlib import Path

# Auto-detect iteris package — works regardless of the Kaggle dataset wrapper folder.
# Expected at /kaggle/input/datasets/anfaalhossain/iteris-package-2/iteris-pkg/iteris
init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris package not attached. Add the iteris-package-2 dataset in the right-hand panel → Datasets.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'

subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet', '--upgrade'], check=True)
print(f'✓ Setup complete. Package at: {PKG_ROOT}')

## 1 · Setup — package on path + config

In [ ]:
import sys
from pathlib import Path

# Auto-detect the iteris package (handles any Kaggle dataset wrapper folder).
init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris package not attached. Add the iteris-package-2 dataset.')
PKG_ROOT = init_files[0].parent.parent
sys.path.insert(0, str(PKG_ROOT))

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config(str(PKG_ROOT / 'configs' / 'camus_lite.yaml'))

# CAMUS data root — auto-detect under /kaggle/input
camus_dirs = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_dirs:
    cfg['data_root'] = str(camus_dirs[0])
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Package root : {PKG_ROOT}')
print(f'Data root    : {cfg["data_root"]}')
print(f'Dataset      : {cfg["dataset"]}')
print(f'Image size   : {cfg["image_size"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train
End-to-end: ingestion → patient-level split → MONAI transforms → cached dataloaders →
Attention Residual U-Net training with cosine schedule + early stopping → best checkpoint saved.

In [ ]:
from iteris.training import run_training

result = run_training(cfg)
model         = result['model']
history       = result['history']
best_dice     = result['best_dice']
test_loader   = result['test_loader']
checkpoint    = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.78)

## 4 · Test-set evaluation
Per-patient Dice + HD95 (pure-torch implementation, no scipy). Writes the CSV that
Week 4 statistical tests and DRL agents consume.

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks
One `.npy` per test sample. These become the DRL environment's `init_mask`.

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON
Snapshot consumed by downstream notebooks.

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)